# 전이학습: 102종 꽃 분류

**30일 딥러닝 완전 정복 · Day 19** — 강의 본문: [https://muy-bien-hwa.github.io/For-my-Claude/](https://muy-bien-hwa.github.io/For-my-Claude/)

### 이 노트북에서 할 일
- 클래스당 학습 이미지 10장뿐인 Flowers102로 세 방법을 비교한다: 처음부터 학습 / 특징 추출 / 미세조정
- 데이터 증강의 효과를 확인한다
- (도전) 직접 찍은 사진으로 나만의 분류기를 만든다

> ⚙️ **런타임 → 런타임 유형 변경 → T4 GPU** 를 먼저 선택하세요. 셀은 위에서부터 `Shift+Enter`로 실행합니다.
> 그래프 라벨은 Colab 기본 글꼴 문제를 피하려고 영어로 표기했습니다.

In [ ]:
import random, os, json, time
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from tqdm.auto import tqdm
device = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def train_one_epoch(model, loader, loss_fn, opt, scheduler=None, clip=None, scaler=None):
    model.train(); tot, correct, n = 0.0, 0, 0
    for xb, yb in tqdm(loader, leave=False):
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=scaler is not None):
            logits = model(xb); loss = loss_fn(logits, yb)
        opt.zero_grad(set_to_none=True)
        if scaler is not None:
            scaler.scale(loss).backward()
            if clip: scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            scaler.step(opt); scaler.update()
        else:
            loss.backward()
            if clip: torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()
        if scheduler is not None: scheduler.step()
        tot += loss.item() * len(yb); correct += (logits.argmax(1) == yb).sum().item(); n += len(yb)
    return tot / n, correct / n

@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval(); tot, correct, n = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb); tot += loss_fn(logits, yb).item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item(); n += len(yb)
    return tot / n, correct / n

def save_ckpt(path, model, opt, epoch, best):
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "epoch": epoch, "best": best}, path)

def load_ckpt(path, model, opt=None):
    ck = torch.load(path, map_location=device)
    model.load_state_dict(ck["model"])
    if opt is not None: opt.load_state_dict(ck["opt"])
    return ck["epoch"], ck["best"]

def fit(model, train_dl, val_dl, epochs, lr=1e-3, wd=0.05, ckpt="best.pt", patience=None, label_smoothing=0.0, amp=True, sgd=False):
    loss_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    if sgd: opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, nesterov=True, weight_decay=wd)
    else:   opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(train_dl), pct_start=0.15)
    scaler = torch.amp.GradScaler("cuda") if (amp and device == "cuda") else None
    hist = {k: [] for k in ["tr_loss", "tr_acc", "va_loss", "va_acc"]}; best, bad = 0.0, 0
    for ep in range(epochs):
        t0 = time.time()
        tl, ta = train_one_epoch(model, train_dl, loss_fn, opt, sched, clip=1.0, scaler=scaler)
        vl, va = evaluate(model, val_dl, nn.CrossEntropyLoss())
        for k, v in zip(hist, [tl, ta, vl, va]): hist[k].append(v)
        print(f"[{ep+1:02d}/{epochs}] train {tl:.4f}/{ta:.3f}  val {vl:.4f}/{va:.3f}  lr {opt.param_groups[0]['lr']:.2e}  {time.time()-t0:.0f}s")
        if va > best: best, bad = va, 0; save_ckpt(ckpt, model, opt, ep, best)
        else:
            bad += 1
            if patience and bad >= patience: print("early stopping"); break
    load_ckpt(ckpt, model)
    return hist

def plot_hist(hist, title=""):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    ax[0].plot(hist["tr_loss"], label="train"); ax[0].plot(hist["va_loss"], label="val"); ax[0].set_title(f"{title} loss"); ax[0].legend()
    ax[1].plot(hist["tr_acc"], label="train"); ax[1].plot(hist["va_acc"], label="val"); ax[1].set_title(f"{title} accuracy"); ax[1].legend()
    plt.show()
print("template ready on", device)

## 1. 데이터
Flowers102의 공식 분할은 학습 1020장(클래스당 10장) / 검증 1020장 / 테스트 6149장입니다. 데이터가 매우 적습니다.

In [ ]:
from torchvision import models
weights = models.ResNet50_Weights.IMAGENET1K_V2
mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)     # 사전학습과 같은 정규화!
aug_tf = transforms.Compose([transforms.RandomResizedCrop(224, scale=(0.5, 1.0)), transforms.RandomHorizontalFlip(),
                             transforms.ColorJitter(0.2, 0.2, 0.2), transforms.ToTensor(), transforms.Normalize(mean, std)])
plain_tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize(mean, std)])
def loaders(train_tf):
    tr = datasets.Flowers102("data", split="train", download=True, transform=train_tf)
    va = datasets.Flowers102("data", split="val", download=True, transform=plain_tf)
    return (DataLoader(tr, batch_size=32, shuffle=True, num_workers=2, drop_last=True),
            DataLoader(va, batch_size=128, num_workers=2))
train_dl, val_dl = loaders(aug_tf)
print(len(train_dl.dataset), len(val_dl.dataset))

## 2. 방법 ① 처음부터 학습 (작은 CNN)

In [ ]:
def small_cnn(n=102):
    def blk(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
    return nn.Sequential(blk(3, 32), blk(32, 64), blk(64, 128), blk(128, 256), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(256, n))
seed_everything(0)
scratch = small_cnn().to(device)
h_scratch = fit(scratch, train_dl, val_dl, epochs=30, lr=3e-3, ckpt="scratch.pt")

## 3. 방법 ② 특징 추출: 백본 동결 + 분류기만

In [ ]:
def resnet50_head(n=102, freeze=True):
    m = models.resnet50(weights=weights)
    if freeze:
        for p in m.parameters(): p.requires_grad = False
    m.fc = nn.Sequential(nn.Dropout(0.2), nn.Linear(m.fc.in_features, n))
    return m.to(device)
seed_everything(0)
fe = resnet50_head(freeze=True)
print("trainable:", sum(p.numel() for p in fe.parameters() if p.requires_grad))
h_fe = fit(fe, train_dl, val_dl, epochs=10, lr=3e-3, ckpt="fe.pt")

## 4. 방법 ③ 미세조정: 분류기 먼저 → layer4 + 분류기를 차등 학습률로

In [ ]:
ft = fe                                                   # 특징 추출로 분류기가 이미 학습된 모델에서 시작
for p in ft.layer4.parameters(): p.requires_grad = True
opt = torch.optim.AdamW([{"params": ft.layer4.parameters(), "lr": 1e-4},
                         {"params": ft.fc.parameters(), "lr": 1e-3}], weight_decay=0.01)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
h_ft = {"va_acc": []}
for ep in range(10):
    tl, ta = train_one_epoch(ft, train_dl, loss_fn, opt)
    vl, va = evaluate(ft, val_dl, nn.CrossEntropyLoss()); h_ft["va_acc"].append(va)
    print(f"[{ep+1}] train {tl:.3f}/{ta:.3f}  val {vl:.3f}/{va:.3f}")

## 5. 결과 비교

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 3.5))
plt.plot(h_scratch["va_acc"], label=f"from scratch (best {max(h_scratch['va_acc']):.2f})")
plt.plot(h_fe["va_acc"], label=f"feature extraction (best {max(h_fe['va_acc']):.2f})")
plt.plot(range(len(h_fe["va_acc"]), len(h_fe["va_acc"]) + len(h_ft["va_acc"])), h_ft["va_acc"], label=f"+ fine-tuning (best {max(h_ft['va_acc']):.2f})")
plt.xlabel("epoch"); plt.ylabel("val acc"); plt.legend(); plt.show()

## 6. 증강의 효과
같은 특징 추출 설정을 증강 없이 학습해 보세요.

In [ ]:
train_dl_noaug, _ = loaders(plain_tf)
seed_everything(0)
fe2 = resnet50_head(freeze=True)
h_noaug = fit(fe2, train_dl_noaug, val_dl, epochs=10, lr=3e-3, ckpt="fe_noaug.pt")
print("with aug:", max(h_fe["va_acc"]), " without aug:", max(h_noaug["va_acc"]))

## 7. (도전) 나만의 사진으로
1. 물건 3종류를 각 30장씩 찍어 Drive의 `MyDrive/my_photos/클래스명/` 폴더에 올립니다.
2. 아래 셀의 경로를 확인하고 실행합니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
root = "/content/drive/MyDrive/my_photos"
if os.path.isdir(root):
    ds = datasets.ImageFolder(root, transform=aug_tf)
    n_val = max(1, len(ds) // 5)
    tr, va = random_split(ds, [len(ds) - n_val, n_val], generator=torch.Generator().manual_seed(0))
    my = resnet50_head(n=len(ds.classes), freeze=True)
    fit(my, DataLoader(tr, 16, shuffle=True), DataLoader(va, 32), epochs=8, lr=3e-3, ckpt="mine.pt")
    print("classes:", ds.classes)
else:
    print("폴더가 없습니다:", root)